# Code Analysis for Hypergraph Proper Coloring

First ensure you can import the main module `src`

In [2]:
import torch
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent.parent))
import src

/home/exs/.conda/envs/kgrouping/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Initialize the environment

We use the `pytorch` framework for deep learning training. PyTorch implements many optimization algorithms behind the scenes to accelerate deep learning training and reduce the use of computing resources, but using optimization algorithms also introduces randomness into the training process. Therefore, if you want to reproduce previous work, you need to ensure that PyTorch has reproducibility support enabled, and all these settings are done in the `init()` function. For more information plz see [Pytorch-Reproducibility](https://docs.pytorch.org/docs/stable/notes/randomness.html)

As previously mentioned, enabling reproducibility will lead to a sharp decline in performance and a significant increase in the use of computing resources. Under our limited resources, we can only provide limited support for reproducibility.

In [3]:
from src import init
init(cuda_index=0, reproducibility=True) # Select cuda:0 device and enable reproducibility

[WARNING] You have enabled the reproducibility feature, which uses a deterministic non-optimized algorithm, greatly affecting the running efficiency
[INFO] Using CUDA device: NVIDIA GeForce RTX 3090 (Index: 0)


## Load Dataset

The following demonstrates the hypergraph proper coloring task using the `cooking200` hypergraph dataset. All datasets used in the paper are stored in the `data/` directory, and of course, for convenience, we have integrated an interface in Python to facilitate the instantiation of text-type datasets, :

In [4]:
from src import Datasets
from src import from_file_to_hypergraph, get_device

data_path = Datasets.Hypergraph_cooking200.path
hg = from_file_to_hypergraph(data_path, True).to(get_device())

Hypergraph(num_v=7403, num_e=2750)


## Define neural network

Of course we need the neural network to conduct deep learning training, In `proper coloring problem on Hypergraph`, We provide a neural network model structure we called `DualHeadNet` on `src/coloring/models`

In [5]:
from src.coloring import DualHeadNet

Next, we need to fill in the specific network layers into the model architecture, here we use `GraphSAGE` as the GNN layer and the `Linear` layer as the fully connected layer. You might wonder why we don't choose `HyperGNN` like `HyperGCN` or `HGNN` for tasks on hypergraphs but instead use `graphGNN`, and how this is achieved ?

In fact, we transformed the hypergraph into a graph using some algorithm for network calculations (you can see [DHG-doc](https://deephypergraph.readthedocs.io/en/latest/tutorial/structure.html#reduced-from-high-order-structures)), but during training, we still used the original hypergraph information in our LOSS FUNCTION. In fact, after our tests, using spatial domain network models like GraphSAGE in coloring problems yields better results.

In [6]:
# In 'LayerType', there are many network layers available for you to choose from, 
# all of which come from the `DHG` library and the `PyG` library
from src import Layer, LayerType 

init_dim = 1024
layers = [
        [Layer(LayerType.GRAPHSAGE, init_dim, 512, hidden_channels=512, num_layers=2, jk="last", drop_rate=0)], # gnn layer
        [],                                                                                                     # shared layer
        [Layer(LayerType.LINEAR, 512, 5, use_bn=True, dropout=0.2)],                                            # coloring layer
        [Layer(LayerType.LINEAR, 512, 5, use_bn=True, dropout=0)],                                              # group layer
]
x = torch.rand(hg.num_v, init_dim)
net = DualHeadNet(layers[0], layers[1], layers[2], layers[3])

## Define Annealing Strategy

You can find a more detailed theoretical description of the `Annealing Strategy` in section 3.4 of the paper.

In [7]:
gini_cons_lambda = lambda e, n: (-200 + 1 * e) / 1000
obj_cof_lambda = lambda e, n: 250 - e / 100
obj_cons_cof_lambda = lambda e, n: 100 - e / 100
cons_cof_lambda = lambda e, n:  5 + e / 1000

## Solving problem

As described in the `README.md`, problems on graphs can be modeled as QUBO problems, whereas problems on hypergraphs can be modeled as PUBO problems. Thus, the current example problem - `hypergraph proper coloring` can be modeled as a PUBO problem.

We provided two interfaces `src/core/run_qubo` and `src/core/run_pubo` to solve these two problems, and each interface provides three specific tasks `maxcut`, `coloring` and `partitioning`

In [8]:
from src import run_pubo

In [ ]:
loss, outs, eval = run_pubo(                    # pubo corresponds to hypergraph
    "coloring",                                 # hypergraph proper coloring task
    net,                                        # Network
    x,                                          # feature for each vertices
    hg,                                         # hypergraph structure
    500,                                        # number of epoch
    4e-4,                                       # learning rate
    "AdamW",                                    # gradient descent algorithm
    clip_grad=True,                             # is gradient clipping enabled
    simple=True,                                # no need to care :)
    gini_cons_cof_lambda=gini_cons_lambda,      # Annealing Strategy 
    obj_cof_lambda=obj_cof_lambda,             
    cons_cof_lambda=cons_cof_lambda,            
    obj_cons_cof_lambda=obj_cons_cof_lambda,    
    evaluate=True                               # enable evaluation mode
)

[INFO] INFO: 0 loops found
[INFO] Hypergraph to graph: 7403 vertices 2715 graph edges
 20%|██        | 102/500 [00:05<00:22, 17.55it/s]

Epoch: 100 | coloring loss: 0.56 | group loss: 5.00 | obj cons: 0.00 | annealing loss: 5204.26


 40%|████      | 200/500 [00:11<00:16, 18.72it/s]

Epoch: 200 | coloring loss: 0.04 | group loss: 5.00 | obj cons: 0.00 | annealing loss: 5313.56


 60%|██████    | 301/500 [00:17<00:14, 13.82it/s]

Epoch: 300 | coloring loss: 0.02 | group loss: 4.22 | obj cons: 0.11 | annealing loss: 8.19


 80%|████████  | 401/500 [00:23<00:06, 15.81it/s]

Epoch: 400 | coloring loss: 0.00 | group loss: 3.00 | obj cons: 0.00 | annealing loss: 2.24


100%|██████████| 500/500 [00:30<00:00, 16.66it/s]


Epoch: 500 | coloring loss: 0.00 | group loss: 3.00 | obj cons: 0.00 | annealing loss: 0.01
+------------[Evaluation Result]------------+
Qualified Edge: 2750/2750 (100.0%)
Color distribution:
[   0    0 3890 1360 2153]
Not converged: 0
Num_Color_used: 3
+-------------------------------------------+


## Evaluation

We have implemented an evaluation function for each task: `src/coloring/coloring_evaluate`, `src/maxcut/maxcut_evaluate` and `src/partitioning/partitioning_evaluate` which can evaluate the results of the model, and you can check some information in the dictionary type it returns.

In [10]:
eval

{'correct_edges': 2750,
 'total_edges': 2750,
 'accuracy': 1.0,
 'color_distribution': array([   0,    0, 3890, 1360, 2153]),
 'not_converged': 0,
 'num_color': 3}

It means that the model's result indicates that all 2750 edges/hyperedges have `2750` edges correctly colored, and the minimum number of colors used is `3`, and there are `0` vertices that do not converge (with a default tolerance of 0.7, meaning results between 0.3 and 0.7 are considered non-convergent and the expectation is {0,1}) 👍

## Using other algorithms for comparison

We use the Tabu and SCIP algorithms for comparison with our algorithm, which respectively implement `src/core/BaseTabuCol` and `src/core/BaseSCIPSolver`. 

### SCIP Solver for coloring

`ColoringSCIPSolver` can simultaneously solve `graph coloring problems` and `hypergraph proper coloring problems`; it only requires a list of edges for a graph/hypergraph

In [11]:
from src.coloring import ColoringSCIPSolver

In [ ]:
css = ColoringSCIPSolver(hg.e[0], 100) 
# data.e[0] is the edge list for a data structure, e[1] is the edge weight for a data structure
# `100` is the upper bound for the number of used colors

### Tabu Solver for coloring

Function `coloring_tabu` can simultaneously solve `graph coloring problems` and `hypergraph proper coloring problems`; it also only requires a list of edges for a graph/hypergraph

In [ ]:
from src.coloring import coloring_tabu

In [ ]:
cts = coloring_tabu('hypergraph', hg.e[0])